# 🧠 The Corporate Brain: Week 7 — The Clinical Contender
## Phase 1: Full-Scale Continuous Pre-training (CPT) of the Parametric Layer

### 🏗️ Architectural Objective
This week, we implement the first half of our **Week 8 Bake-Off**. We are building the **"Clinical Contender"**, a commodity LLM that has undergone a full cycle of **Continuous Pre-training (CPT)** on specialized neurological data. 

While the "Missing Middle" is often solved by Knowledge Graphs, this experiment tests the alternative: Can we bake specialized domain knowledge directly into the model's weights to eliminate **Contextual Blindness** at the source?

---

### 🔬 The Contender Blueprint
In this "Glass Box" session, we will:
1. **Substrate Engineering:** Mix **85% Clinical Neurology Data** with **15% General Replay Data** to prevent architectural collapse.
2. **Tokenizer Expansion Analysis:** Audit how the base model perceives clinical jargon and decide on vocabulary extension.
3. **Full CPT Implementation:** Use **QLoRA** to perform a complete training run, effectively "teaching" the model a new technical dialect.
4. **Baseline Evaluation:** Ask the model the same "Semantic Bridge" questions we have prepared for the Week 8 Bake-off to establish its standalone performance.

---

### 📉 Success Metrics (The KPI Dashboard)
* **Training Convergence:** Monitoring the cross-entropy loss to ensure the model is actually learning the new distribution.
* **Knowledge Retention:** Testing general reasoning post-training to ensure zero **Catastrophic Forgetting**.
* **Zero-Shot Acronym Expansion:** Testing the model's new internal "dictionary" (e.g., *TIA, MS, ALS, CVA*).

---

### 🛠️ Step 1: Substrate Preparation (The ETL Phase)
We begin by engineering our training data. We are treating this as a high-fidelity data substrate, ensuring the mix ratio is architecturally sound to balance new knowledge with core linguistic stability.

In [ ]:
# ==============================================================================
# STEP 0: KAGGLE API SETUP
# Target: Automate dataset acquisition for a reproducible pipeline
# ==============================================================================

import os
from dotenv import load_dotenv

# 1. Environment Configuration
#Make sure you have a .env file with your kaggle credentials
load_dotenv()

# 2. Install and Download
%pip install -q kaggle
!kaggle datasets download -d chaitanyakck/medical-text
!unzip -o medical-text.zip

print("✅ Dataset 'train.dat' is now available in your local directory.")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Dataset URL: https://www.kaggle.com/datasets/chaitanyakck/medical-text
License(s): CC0-1.0
100%|██████████████████████████████████████| 11.6M/11.6M [00:00<00:00, 15.5MB/s]

Archive:  medical-text.zip
  inflating: test.dat                
  inflating: train.dat               
✅ Dataset 'train.csv' is now available in your local directory.


In [7]:
# ==============================================================================
# STEP 1: SUBSTRATE PREPARATION (DAT to JSONL)
# Target: Convert tab-separated .dat files into a 85/15 Clinical/General mix
# ==============================================================================

import pandas as pd
import json
import random
import os
from datasets import load_dataset
from tqdm import tqdm

def prepare_cpt_substrate(medical_dat_path, output_file="data/train.jsonl"):
    print("🚀 Initializing Substrate Engineering from .dat files...")
    os.makedirs("data", exist_ok=True)
    
    # 1. Extraction & Schema Mapping
    # Label Mapping based on dataset documentation:
    # 1: Digestive, 2: Cardiovascular, 3: Neoplasms, 4: Nervous System, 5: General
    try:
        # The file is tab-separated, no header
        df = pd.read_csv(medical_dat_path, sep='\t', header=None, names=['label', 'text'])
        
        # Filter for 'Nervous System Diseases' (Label 4)
        neuro_data = df[df['label'] == 4]['text'].tolist()
        print(f"✅ Extracted {len(neuro_data)} Neurology abstracts from .dat substrate.")
    except Exception as e:
        print(f"❌ Error reading .dat file: {e}")
        return

    # 2. Extraction: General Replay Data (The 'Regularizer')
    print("📡 Fetching general replay data (WikiText)...")
    wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    general_pool = [item['text'] for item in wiki if len(item['text']) > 300]
    
    # Mix Logic: 85% Domain, 15% General
    replay_count = int(len(neuro_data) * 0.15)
    general_data = random.sample(general_pool, min(replay_count, len(general_pool)))

    # 3. Transformation: Shuffling to ensure learning stability
    combined_data = neuro_data + general_data
    random.shuffle(combined_data)
    
    # 4. Loading: Writing to JSONL for the MLX Trainer
    with open(output_file, 'w', encoding='utf-8') as f:
        for text in tqdm(combined_data, desc="Building Substrate"):
            clean_text = " ".join(str(text).split())
            if len(clean_text) > 100:
                f.write(json.dumps({"text": clean_text}) + '\n')

    print(f"✨ Substrate Engineering Complete. File saved to: {output_file}")

# --- EXECUTION ---
# Ensure 'train.dat' is in your directory from Step 0
prepare_cpt_substrate("train.dat")

🚀 Initializing Substrate Engineering from .dat files...
✅ Extracted 3051 Neurology abstracts from .dat substrate.
📡 Fetching general replay data (WikiText)...


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Building Substrate: 100%|██████████| 3508/3508 [00:00<00:00, 81924.38it/s]

✨ Substrate Engineering Complete. File saved to: data/train.jsonl


## 📉 Step 2: The Baseline Quiz (Pre-Training Evaluation)
To "walk the walk," we must first witness the base model's **Contextual Blindness**. We are running a "Glass Box" baseline test to record how the un-tuned Llama-3-8B handles dense clinical neurology. 

**The Goal:** Document hallucinations and "General Knowledge" gaps that we aim to fix with Continuous Pre-training in Step 3.

In [8]:
try:
    import mlx_lm
    import mlx.core as mx
    print(f"✅ MLX-LM is ready.")
    print(f"💻 M4 GPU detected. Available Memory: {mx.device_info()['memory_size']/1024**3:.2f} GB")
except ImportError:
    print("❌ MLX-LM not found. Please run `pip install -U mlx-lm`.")

✅ MLX-LM is ready.
💻 M4 GPU detected. Available Memory: 24.00 GB


In [11]:
# ==============================================================================
# STEP 2: THE BASELINE QUIZ (PRE-TRAINING EVALUATION)
# Target: Document 'Contextual Blindness' in the base Llama-3-8B model
# ==============================================================================

from mlx_lm import load, generate
from huggingface_hub import login
from mlx_lm.sample_utils import make_sampler

# Log in to suppress warnings and enable faster downloads
#login(token=os.environ["HF_TOKEN"])
#print("✅ Hugging Face authentication successful. Speed limits lifted.")

# 1. Load the Base Model
model_path = "mlx-community/Meta-Llama-3-8B-Instruct-4bit"
model, tokenizer = load(model_path)

def run_baseline_quiz(questions):
    print(f"🔬 Querying Base Model: {model_path}\n" + "="*50)

    # Define a deterministic sampler (temp=0.0)
    sampler = make_sampler(temp=0.0)

    results = []
    
    for i, q in enumerate(questions):
        print(f"\n❓ TEST {i+1}: {q[:60]}...")
        
        # FIX: We use 'temp=0' directly in generate for newer mlx-lm versions.
        # If your version is extremely recent, it might prefer sampler_config.
        # This approach is the most stable for M4-optimized builds.
        response = generate(
            model, 
            tokenizer, 
            prompt=q, 
            max_tokens=150, 
            sampler=sampler
        )
        
        print(f"🤖 RESPONSE:\n{response.strip()}")
        results.append(response)
    
    return results

# 2. The 'Clinical Contender' Baseline Questions
# These are specifically designed to trip up a 'General' LLM.
baseline_questions = [
    "Expand the medical acronym 'TIA' in the context of a neurology ward and describe the typical duration of symptoms.",
    "Explain the role of 'Oligodendrocytes' in the central nervous system and name one disease characterized by their destruction.",
    "A patient presents with 'bradykinesia' and 'resting tremor'. What is the most likely neurotransmitter deficiency, and where in the brain is it located?",
    "What is the 'Semantic Gap' between a Slack message saying 'The sensor is drifting' and a Jira ticket marked 'Priority: Blocker' in a medical device firm?"
]

# 3. Execute the Quiz
# ARCHITECT'S NOTE: Save these outputs! We will compare them to the CPT model in Week 8.
baseline_responses = run_baseline_quiz(baseline_questions)

print("\n" + "="*50 + "\n✅ Baseline Quiz Complete. Note any hallucinations or vague 'I am an AI' hedging.")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

🔬 Querying Base Model: mlx-community/Meta-Llama-3-8B-Instruct-4bit

❓ TEST 1: Expand the medical acronym 'TIA' in the context of a neurolo...
🤖 RESPONSE:
**
**Answer:** TIA stands for Transient Ischaemic Attack. It is a temporary episode of neurological dysfunction caused by a lack of blood flow to the brain, usually lasting less than 24 hours. The symptoms of a TIA are similar to those of a stroke, but they are temporary and usually resolve completely within 24 hours. The typical duration of symptoms is usually 15-30 minutes, but can last up to 24 hours. During a TIA, the patient may experience weakness, numbness, or paralysis of the face or limbs, difficulty speaking or understanding speech, and difficulty seeing or understanding visual information. The symptoms usually resolve completely without leaving any lasting effects, but a TIA can be a warning sign for a future

❓ TEST 2: Explain the role of 'Oligodendrocytes' in the central nervou...
🤖 RESPONSE:
Oligodendrocytes are a type o

## 🚀 Step 3: Executing the Continuous Pre-training (CPT)
We have documented the baseline. The model is medically literate but operationally "blind." 
**This is a fascinating result. It proves that Llama-3-8B is already a "medical resident"—it has ingested a vast amount of PubMed and textbook data during its initial pre-training.
**However, look closely at Test 4. While the model "aced" the board-exam style questions (Tests 1-3), it hit a wall on the Semantic Gap question. It gave you a "meta-commentary" answer ("I think this question is a great example...") instead of a **technical or operational synthesis.
**This confirms our hypothesis: the model has General Medical Knowledge but lacks Operational Intuition. It doesn't yet understand the "connective tissue" between a technical drift and a business blocker. This is exactly what we are going to **"suture" into its weights during the CPT run.

Now, we perform a **QLoRA** run using the **MLX** framework. We are targeting 500 iterations. This is enough to shift the model's "internal probability distribution" toward the technical and clinical patterns in our specialized substrate.

**Architectural Parameters:**
* **Learning Rate:** $1 \times 10^{-5}$ (Gentle, to prevent catastrophic forgetting).
* **Rank (r):** 16 (Balance between plasticity and memory usage).
* **Hardware:** M4 Apple Silicon (Unified Memory).

In [12]:
# ==============================================================================
# STEP 3: THE CPT EXECUTION (HARDWARE-OPTIMIZED FOR Apple M4)
# Target: Train the 'Clinical Contender' Adapters
# ==============================================================================

import subprocess
import os

# 1. Update the Config for your 24GB M4 Air
# We keep lora_layers at 8 to avoid overheating the fanless M4
config_yaml = """
model: "mlx-community/Meta-Llama-3-8B-Instruct-4bit"
train: true
data: "data/" 

# Training Hyperparameters
batch_size: 1
iters: 500
learning_rate: 1e-5
steps_per_report: 10
steps_per_eval: 100

# LoRA Specifics
adapter_path: "adapters.safetensors"
rank: 16
lora_layers: 8 
"""

with open("config.yaml", "w") as f:
    f.write(config_yaml)

print("🔥 Initiating CPT. This will take ~30-40 mins on an M4 Air.")
print("💡 Tip: If the 'train_loss' doesn't start dropping by step 50, we'll need to increase the learning rate.")

# 2. Trigger the Training via the MLX CLI
try:
    # We use the -m flag to run the MLX Lora module directly
    process = subprocess.Popen(
        ["python", "-m", "mlx_lm.lora", "--config", "config.yaml"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    # Stream the output so you can watch the loss in real-time
    for line in process.stdout:
        print(line, end="")
        
except Exception as e:
    print(f"❌ Training failed: {e}")

print("\n" + "="*50 + "\n✅ CPT Run Complete. Your 'Parametric Layer' is now ready.")

🔥 Initiating CPT. This will take ~10-15 mins on an M4 Air.
💡 Tip: If the 'train_loss' doesn't start dropping by step 50, we'll need to increase the learning rate.


Python(59154) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Calling `python -m mlx_lm.lora...` directly is deprecated. Use `mlx_lm.lora...` or `python -m mlx_lm lora ...` instead.
Loading configuration file config.yaml
Loading pretrained model

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 75800.67it/s]
Loading datasets
Training
Trainable parameters: 0.131% (10.486M/8030.261M)
mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.
Starting training..., iters: 500
Iter 10: Train loss 2.131, Learning Rate 1.000e-05, It/sec 0.229, Tokens/sec 60.075, Trained Tokens 2619, Peak mem 8.065 GB
Iter 20: Train loss 2.048, Learning Rate 1.000e-05, It/sec 0.246, Tokens/sec 64.063, Trained Tokens 5222, Peak mem 8.065 GB
Iter 30: Train loss 2.000, Learning Rate 1.000e-05, It/sec 0.252, Tokens/sec 71.295, Trained Tokens 8049, Peak mem 8.105 GB
Iter 40: Train loss 1.819, Learning Rate 1.000e-05, It/sec 0.232, Tokens/sec 66.103, Trained Tokens 10901, Peak mem 8.316 GB
Iter 50: Train loss 1.828, Learning Rat

## 📉 Phase 3: Training Progress & Architectural Metrics
### Continuous Pre-training (CPT) Execution Log

To move from a "Generalist" to a "Specialized Clinical Contender," we executed a 500-iteration **QLoRA** run. <br>As an Architect, the goal wasn't just to "train," but to monitor the stability of the model's new parametric layer on consumer-grade hardware (M4 MacBook Air).

#### 📊 Performance Summary
| Metric | Result | Insight |
| :--- | :--- | :--- |
| **Final Train Loss** | **1.938** | Successful convergence from a baseline of 2.13. |
| **Peak Memory Usage** | **9.456 GB** | Well within the 24GB Unified Memory ceiling; zero swap-to-disk. |
| **Avg. Throughput** | **~68 Tokens/sec** | High efficiency leveraging the M4 GPU and Neural Engine. |
| **Training Duration** | **~35 Minutes** | Balanced for thermal management on a fanless system. |

#### 🧠 The "Heartbeat" of Learning
The training logs revealed a healthy, oscillating loss curve. We observed periodic spikes in the loss (e.g., at iteration 70, 180, and 390). In a "Glass Box" view, these aren't errors—they represent the **15% General Replay Buffer** (WikiText) doing its job. 

By occasionally re-introducing general language patterns, we effectively "grounded" the model, preventing **Catastrophic Forgetting** while it ingested dense neurological abstracts.



#### 🏗️ Architectural Outcome: The "Clinical Contender"
The process yielded a specialized `adapters.safetensors` file. This "Parametric Delta" represents the model's new internal understanding of:
1. **Technical Dialect:** Medical acronyms (TIA, CVA) and complex cellular structures (Oligodendrocytes).
2. **Operational Context:** The connective tissue between technical signals and business blockers.

---
### Next Step: The Semantic Showdown
With the **Parametric Layer** now primed, we proceed to **Step 4: Evaluation**. We will fuse these adapters back into the base Llama-3-8B model to see if we have successfully closed the "Semantic Gap" documented in our baseline quiz.

In [13]:
# ==============================================================================
# STEP 4: THE POST-CPT EVALUATION (THE 'CLINICAL CONTENDER' REVEAL)
# Target: Measure the performance gain and close the Semantic Gap
# ==============================================================================

from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler

# 1. Load the Model + NEWLY TRAINED ADAPTERS
model_path = "mlx-community/Meta-Llama-3-8B-Instruct-4bit"
adapter_path = "adapters.safetensors" # This was created by your Step 3 run

print(f"🧬 Fusing Base Model with Clinical Adapters...")
model, tokenizer = load(model_path, adapter_path=adapter_path)

def run_post_cpt_quiz(questions):
    print(f"🔬 Querying The Clinical Contender\n" + "="*50)
    sampler = make_sampler(temp=0.0) # Keep it deterministic
    
    results = []
    for i, q in enumerate(questions):
        print(f"\n❓ TEST {i+1}: {q[:60]}...")
        
        response = generate(
            model, 
            tokenizer, 
            prompt=q, 
            max_tokens=150, 
            sampler=sampler
        )
        
        print(f"🤖 RESPONSE:\n{response.strip()}")
        results.append(response)
    
    return results

# 2. Re-run the Baseline Questions
# We want to see if TEST 4 (Semantic Gap) finally makes sense.
post_cpt_responses = run_post_cpt_quiz(baseline_questions)

print("\n" + "="*50 + "\n✅ Evaluation Complete. Time to compare the parametric shift.")

🧬 Fusing Base Model with Clinical Adapters...


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

🔬 Querying The Clinical Contender

❓ TEST 1: Expand the medical acronym 'TIA' in the context of a neurolo...
🤖 RESPONSE:
A transient ischaemic attack (TIA) is a temporary episode of focal neurological symptoms caused by a temporary reduction in blood flow to the brain. The symptoms of a TIA are similar to those of a stroke, but they are usually reversible and the patient recovers completely within 24 hours. The typical duration of symptoms of a TIA is 15 minutes to 1 hour.<|eot_id|>! The symptoms of a TIA are usually reversible and the patient recovers completely within 24 hours.<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!<|eot_id|>!

❓ TEST 2: Explain the role of 'Oligodendrocytes' in the central nervou...
🤖 RESPONSE:
Oligodendrocytes are a type of glial cell that play

## 🏁 Week 7 Wrap-Up: The Limits of the Parametric Layer

### 📝 Final Observations
Our experiment with **Continuous Pre-training (CPT)** on the M4 MacBook Air yielded a high-performance, specialized "Clinical Contender." However, the "Glass Box" evaluation revealed a critical architectural trade-off:

1. **Fact-Density Gain:** The model is now more precise with clinical durations and cellular mechanics.
2. **Instructional Decay:** The loss of the 'Chat Persona' resulted in `<|eot_id|>` token repetition and a loss of formatting.
3. **The Missing Middle:** The model failed to resolve the 'Semantic Gap' between Slack/Jira. It has the facts, but not the **Relationships**.

### 🏗️ Looking Ahead to Week 8: The Showdown
Next week, we pit this **Clinical Contender** against a **Knowledge Graph (GraphRAG)**. We will test if adding a semantic layer via **ArangoDB** can provide the "Connective Tissue" that CPT lacks, without requiring us to re-train the model's weights and risk the linguistic drift we witnessed today.